# Notebook 07 — CLIP ViT-B/32 Multimodal Embeddings

**Phase 4 · Task group 117.** Goal: embed every figure from `figures_metadata.parquet` using **openai/clip-vit-base-patch32** (image + text towers), persist to a tier-namespaced Chroma collection, and verify that cross-modal retrieval works — i.e. a natural-language query finds the right figure.

### Why CLIP
CLIP was trained to align image and text in a shared 512-d space. That means:
* `embed_text("CT chest scan showing ground-glass opacities")` lives near
* `embed_image(chest_CT.png)` in the same vector space.

So we can reuse the same retrieval pipeline (Chroma k-NN) for both modalities — we just change which embedder produced the query vector.

### Why a separate collection
The text corpus lives in `sciret_<tier>_bge_m3_cs400_o50` (dim 1024); figures live in `sciret_<tier>_clip_vitb32` (dim 512). Chroma collections can only hold one embedding dim, so the multimodal router in Notebook 08 queries both collections and fuses the results.

### What this ships
1. CLIP loaded + `CLIP_DIM == 512` asserted.
2. Figure embeddings cached to parquet, then loaded into Chroma.
3. Cross-modal retrieval test — 5 text queries → top-3 figures each.
4. UMAP of the joint CLIP space with text queries plotted alongside figures.


In [1]:
import os, sys, json, time, warnings
from pathlib import Path

NB_DIR = Path.cwd()
PROJECT_ROOT = NB_DIR.parent if (NB_DIR.parent / "2_src").exists() else NB_DIR
sys.path.insert(0, str(PROJECT_ROOT / "2_src"))

os.environ.setdefault("SCIRET_TIER", "tier1")
from config import get_config, CLIP_MODEL, CLIP_DIM, SEED
CFG = get_config()
print(CFG.summary())

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid"); warnings.filterwarnings("ignore")


[SciRet:tier1] size=1000 root=D:\SciRet-Scientific-Information-Made-Easy\Sciret2 chunks=chunks.parquet chroma=chroma_db/sciret_tier1_bge_m3_cs400_o50


In [2]:
FIG_PATH = CFG.figures_metadata_path
if not FIG_PATH.exists():
    print("figures_metadata.parquet missing — run Notebook 06 first.")
df_figs = pd.read_parquet(FIG_PATH) if FIG_PATH.exists() else pd.DataFrame()
print("figures:", len(df_figs))
df_figs.head(3)


figures: 0


,figure_id,cord_uid,page,img_idx_on_page,asset_path,caption_raw,width,height,caption


## 1. Load CLIP ViT-B/32

In [4]:
import torch
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", DEVICE)

clip = CLIPModel.from_pretrained(CLIP_MODEL).to(DEVICE).eval()
proc = CLIPProcessor.from_pretrained(CLIP_MODEL)

# Sanity probe.
with torch.no_grad():
    t_in = proc(text=["covid lung CT"], return_tensors="pt", padding=True).to(DEVICE)
    out = clip.get_text_features(**t_in)
    # handle both tensor and ModelOutput return types
    v = (out if isinstance(out, torch.Tensor) else out.pooler_output).cpu().numpy()
assert v.shape[1] == CLIP_DIM, f"expected {CLIP_DIM}-d, got {v.shape[1]}"
print("CLIP ok — dim =", v.shape[1])


device: cuda


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 28893.69it/s]
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CLIP ok — dim = 512


## 2. Embed figures (cache-or-build)

In [5]:
CACHE = CFG.embeddings_dir / f"clip_figures_{CFG.tier}.parquet"

def normalize(x):
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + 1e-9)

def embed_images(paths, batch=16):
    outs = []
    for i in range(0, len(paths), batch):
        imgs = []
        for p in paths[i:i+batch]:
            try:
                imgs.append(Image.open(p).convert("RGB"))
            except Exception:
                imgs.append(Image.new("RGB", (224,224), "white"))
        with torch.no_grad():
            inp = proc(images=imgs, return_tensors="pt").to(DEVICE)
            v = clip.get_image_features(**inp).cpu().numpy()
        outs.append(v)
        if (i//batch) % 10 == 0:
            print(f"  img {min(i+batch,len(paths))}/{len(paths)}")
    V = np.vstack(outs).astype("float32") if outs else np.zeros((0,CLIP_DIM),"float32")
    return normalize(V)

def embed_texts(texts, batch=32):
    outs = []
    for i in range(0, len(texts), batch):
        with torch.no_grad():
            inp = proc(text=texts[i:i+batch], return_tensors="pt",
                       padding=True, truncation=True, max_length=77).to(DEVICE)
            v = clip.get_text_features(**inp).cpu().numpy()
        outs.append(v)
    V = np.vstack(outs).astype("float32") if outs else np.zeros((0,CLIP_DIM),"float32")
    return normalize(V)

if CACHE.exists():
    emb_df = pd.read_parquet(CACHE)
    print(f"cache HIT  {CACHE}  ({len(emb_df):,})")
elif len(df_figs):
    img_vecs = embed_images(df_figs["asset_path"].tolist())
    txt_vecs = embed_texts(df_figs["text_for_embedding"].fillna("").tolist()) if "text_for_embedding" in df_figs else np.zeros_like(img_vecs)
    emb_df = pd.DataFrame({
        "figure_id": df_figs["figure_id"].tolist(),
        "image_vector": list(img_vecs),
        "caption_vector": list(txt_vecs),
    })
    emb_df.to_parquet(CACHE, index=False)
    print(f"cache MISS  saved  {CACHE}")
else:
    emb_df = pd.DataFrame(columns=["figure_id","image_vector","caption_vector"])


## 3. Persist to the figures Chroma collection

In [6]:
import chromadb
client = chromadb.PersistentClient(path=str(CFG.chroma_dir))
fig_col = CFG.figure_collection

try:
    col = client.get_collection(fig_col)
    if col.count() == len(emb_df):
        print(f"fig collection already populated: {col.count()}")
    else:
        client.delete_collection(fig_col)
        col = client.create_collection(fig_col, metadata={"hnsw:space":"cosine"})
except Exception:
    col = client.create_collection(fig_col, metadata={"hnsw:space":"cosine"})

if col.count() == 0 and len(emb_df):
    BATCH = 256
    ids = emb_df["figure_id"].tolist()
    metas = df_figs.set_index("figure_id").loc[ids][
        ["cord_uid","asset_path","text_for_embedding"]
    ].rename(columns={"text_for_embedding":"caption"}).to_dict("records")
    img_v = np.asarray(emb_df["image_vector"].tolist(), dtype="float32")
    docs  = df_figs.set_index("figure_id").loc[ids, "text_for_embedding"].fillna("").tolist()
    for i in range(0, len(ids), BATCH):
        col.add(
            ids=ids[i:i+BATCH],
            embeddings=img_v[i:i+BATCH].tolist(),
            metadatas=metas[i:i+BATCH],
            documents=docs[i:i+BATCH],
        )
    print("populated fig collection:", col.count())


## 4. Cross-modal retrieval test

5 text queries → top-3 figures each. We print the matched filenames + captions and the cosine similarity; scores >0.20 indicate plausible alignment for ViT-B/32.

In [7]:
QUERIES = [
    "CT scan showing ground-glass opacities in COVID-19 pneumonia",
    "Kaplan-Meier survival curve of ICU patients",
    "SARS-CoV-2 spike protein structural diagram",
    "bar chart of vaccine efficacy by age group",
    "gel electrophoresis of PCR products",
]

def retrieve_figs(q, k=3):
    qv = embed_texts([q])[0].tolist()
    res = col.query(query_embeddings=[qv], n_results=k, include=["distances","documents","metadatas"])
    return [
        {"figure_id": i, "sim": round(1.0-d, 3),
         "caption": doc[:120], "asset": m.get("asset_path","")}
        for i,d,doc,m in zip(res["ids"][0], res["distances"][0],
                              res["documents"][0], res["metadatas"][0])
    ]

if col.count():
    for q in QUERIES:
        print("Q:", q)
        for r in retrieve_figs(q):
            print(f"  {r['sim']:.3f}  {r['figure_id']}  | {r['caption']}")
        print()
else:
    print("no figures in collection — run Notebook 06 with real PDFs first.")


no figures in collection — run Notebook 06 with real PDFs first.


## 5. Joint-space UMAP

We embed the same queries into CLIP-text space, project the joint (figures ∪ queries) set into 2D, and render — a well-aligned space shows each query landing near its retrieved figures.

In [8]:
if len(emb_df) > 10:
    img_v = np.asarray(emb_df["image_vector"].tolist(), dtype="float32")
    q_v   = embed_texts(QUERIES)

    try:
        import umap
        reducer = umap.UMAP(n_components=2, random_state=SEED, metric="cosine", n_neighbors=15, min_dist=0.2)
        name = "UMAP"
    except Exception:
        from sklearn.decomposition import PCA
        reducer = PCA(n_components=2, random_state=SEED); name = "PCA"

    joint = np.vstack([img_v, q_v])
    coords = reducer.fit_transform(joint)
    img_c = coords[:len(img_v)]
    q_c   = coords[len(img_v):]

    fig, ax = plt.subplots(figsize=(7,6))
    ax.scatter(img_c[:,0], img_c[:,1], s=8, alpha=0.5, color="#64748b", label="figures")
    ax.scatter(q_c[:,0], q_c[:,1], s=80, marker="*", color="#dc2626", label="text queries")
    for i, q in enumerate(QUERIES):
        ax.annotate(f"Q{i+1}", (q_c[i,0], q_c[i,1]), fontsize=9,
                    xytext=(5,5), textcoords="offset points")
    ax.set_title(f"CLIP joint image/text space — {name}")
    ax.set_xticks([]); ax.set_yticks([]); ax.legend()
    fig.tight_layout()
    fig.savefig(CFG.results_dir / "clip_joint_space_umap.png", dpi=140)
    plt.show()


---
**Outputs**
* `1_data/embeddings/<tier>/clip_figures_<tier>.parquet`
* `1_data/embeddings/<tier>/chroma_db/sciret_<tier>_clip_vitb32` (figures collection)
* `4_results/<tier>/clip_joint_space_umap.png`

**Next:** Notebook 08 — route text + figures through one multimodal pipeline, generate grounded answers.
